# Speech-to-speech

Great question! Let me show you how the architecture changes when using the local Hugging Face GPT-2 model instead of Claude API:Perfect! Here are the **key differences** between Claude API vs Local GPT-2 architectures:

## 🔄 **Major Architecture Changes:**

### **1. Model Initialization (New Step)**
- **GPT-2**: Requires loading 774M parameters (~3GB) into memory
- **Claude**: No local model - just API client initialization
- **Impact**: 30-60 second startup time for GPT-2

### **2. Text Processing Location**
- **GPT-2**: All inference happens locally on Colab GPU/CPU
- **Claude**: Text sent to Anthropic servers via HTTPS
- **Impact**: Privacy vs computational resources

### **3. Response Generation Process**
- **GPT-2**: Multi-step tokenization → inference → decoding
- **Claude**: Single API call with JSON response
- **Impact**: More complex error handling for GPT-2

## ⚖️ **Trade-offs Comparison:**

| Aspect | Claude API | Local GPT-2 |
|--------|------------|-------------|
| **Startup Time** | Instant | 30-60 seconds |
| **Memory Usage** | ~50MB | ~3GB |
| **Response Quality** | Superior | Good |
| **Cost** | Pay per request | Free (after loading) |
| **Privacy** | Data sent to API | Fully local |
| **Reliability** | Depends on internet | Offline capable |
| **Scalability** | Unlimited | Limited by hardware |

## 🎯 **For Your AI/ML Background:**

**Claude API Approach:**
- Better for production/demos
- Consistent performance
- Easy to swap models (GPT-4, etc.)

**Local GPT-2 Approach:**
- Better for research/experimentation
- Full control over parameters
- Good for understanding transformer internals
- Can fine-tune on your trading data

The **Claude version is cleaner** from an architecture standpoint - simpler pipeline, better separation of concerns, and more reliable. The **GPT-2 version gives you more control** but adds complexity in tokenization, memory management, and error handling.

Which approach do you prefer for your use case?

## System Diagram

with `openai` open source model on huggingface

In [1]:
import base64
import sys
import json
import zlib
from IPython.display import SVG, Image
import requests

mermaid_code = '''
sequenceDiagram
    participant U as User/Browser
    participant G as Gradio Interface
    participant VP as VoiceProcessor
    participant GPT2 as GPT-2 Model<br/>(Local HuggingFace)
    participant TOK as Tokenizer<br/>(GPT-2 Tokenizer)
    participant GPU as GPU/CPU<br/>(Colab Runtime)
    participant SR as Speech Recognition<br/>(Google API)
    participant TTS as Text-to-Speech<br/>(gTTS)
    participant FS as File System

    Note over U,FS: Voice-to-Voice with Local GPT-2

    rect rgb(255, 240, 240)
        Note over VP,GPU: Model Initialization (One-time)
        VP->>TOK: Load GPT-2 tokenizer
        VP->>GPT2: Load GPT-2-large model (774M params)
        GPT2->>GPU: Move model to GPU/CPU
        Note over VP,GPU: ~3GB model loaded into memory<br/>Takes 30-60 seconds on first run
    end

    U->>G: 1. Record voice via microphone
    Note over U,G: Audio Format: WAV/WebM<br/>Sample Rate: ~44kHz

    G->>VP: 2. Pass audio file path
    Note over G,VP: File type: filepath<br/>Temporary file location

    VP->>SR: 3. Convert audio to WAV format
    VP->>SR: 4. Transcribe audio to text
    SR->>VP: 5. Return transcribed text
    Note over SR,VP: Uses Google's speech-to-text<br/>Same as Claude version

    alt Transcription successful
        VP->>TOK: 6. Create conversation prompt
        Note over VP,TOK: Format: "Human: {text} AI:"

        TOK->>VP: 7. Tokenize input text
        Note over TOK,VP: Convert text to token IDs<br/>Add special tokens

        VP->>GPT2: 8. Generate response tokens
        Note over VP,GPT2: Parameters:<br/>max_length: 80<br/>temperature: 0.7<br/>top_p: 0.85

        GPT2->>GPU: 9. Run inference locally
        Note over GPT2,GPU: Forward pass through model<br/>Sampling from probability distribution

        GPU->>GPT2: 10. Return generated tokens
        GPT2->>VP: 11. Return token sequence

        VP->>TOK: 12. Decode tokens to text
        TOK->>VP: 13. Return AI response text
        Note over TOK,VP: Extract AI portion only<br/>Clean up formatting

        VP->>TTS: 14. Convert response to speech
        TTS->>FS: 15. Generate audio file
        FS->>VP: 16. Return audio file path

        VP->>G: 17. Return audio + text
        G->>U: 18. Play audio response

    else Transcription failed
        VP->>TTS: 6b. Generate error message
        TTS->>VP: 7b. Return error audio
        VP->>G: 8b. Return error response
        G->>U: 9b. Play error message
    end

    Note over U,FS: Local Processing vs API Call<br/>Voice → Text → Local AI → Text → Voice
'''

def js_btoa(data):
    return base64.b64encode(data)

def pako_deflate(data):
    compress = zlib.compressobj(9, zlib.DEFLATED, 15, 8, zlib.Z_DEFAULT_STRATEGY)
    compressed_data = compress.compress(data)
    compressed_data += compress.flush()
    return compressed_data

def genPakoLink(graphMarkdown: str):
    jGraph = {"code": graphMarkdown, "mermaid": {"theme": "standard"}}
    byteStr = json.dumps(jGraph).encode('utf-8')
    deflated = pako_deflate(byteStr)
    dEncode = js_btoa(deflated)
    link_code = dEncode.decode('ascii')
    return link_code

mermaid_link = genPakoLink(mermaid_code)
print("mermaid.live link:")
print('http://mermaid.live/edit#pako:' + mermaid_link)

mermaid.live link:
http://mermaid.live/edit#pako:eNqFVl1v6jgQ/SsjXrZXWyhQ+oVWlbh0odVu90YEuC9IVyYZgtXEztpOW1p1f/uO7QRS4O7y0EIyZz7OnBn7vRHJGBt9aCyExr8LFBHecZYoli0E0CdnyvCI50wYmAHTMNOozr4q+UL/D03G1mSsWMwlPAiDasUiPDSbB9ZuLnmEgZIRai2POQumXecvmDa78EiJpr8t1dntyZ8yYincF0nCRTKiCF8OwdNvf1jsVD6h4G+oPNK72j78cizozMecnQ2DmUcNZcqWMCmE4dmxWOHEQsIcMVrDBCOZCG64FGVMKZMUYRA8HEtzGro08dU0jWx6Hx6X0LsjiJEDjDi5DDfaIDXKG/0lDYJ8RgWz01HY9/Rap+4LvHCzBs+cY6GCKYwMqGR50r24OIVur+3+lIE/+50Hp8RL37eC+ktFspS/MVsrnHwTFG1HkP3Mg+btLXWiT4FZXDbSVOzv2dl21w2bKVMJQuaCnVxd9R4tDyzTtQAW46Auq+fK2siqgf9Vxj/n468lIKWoGAMXhMwwk2rjejBlT6jhvN28bIOmvopYA5W64koTZ0XJIIq4InNmk+lDp+VkoGJ4dtw/cwYZj5TM11LgYbsIMijszIykypjpw/fB/Ow7Lh9dFiHLcur2hBmkpHu9p/u3Kt6Y4s2DPnRbEDCtgTkvKyuOnJn1fqTxqTV22jGbnLxZS2voq8Usl4qpjXeQklZsZ6tYrknhpA/nLRhKQf5MGY9Io4Rh5ZLfM+61YKqY0JHiS9zZGxK8twwnZQ0XljRTKAFmC4hrhrsywomrg1YRDaqbrl80aDc6Vu8WUhGHdliGKcWlLqDStXpYarap5U7CuojsLloV6TEJX1LZCqkJELnqtdd9rmSWm5/ozAGrpi4a90XGRB/ebYYfMHjoLxpVNvZD1iUXV63tkiJR5oWp8fA5CGEcGVVHrJ0j2MLh4U4

with Anthropic Claude (requires API Key)

In [2]:
import base64
import sys
import json
import zlib
from IPython.display import SVG, Image
import requests

mermaid_code = '''
sequenceDiagram
    participant U as User/Browser
    participant G as Gradio Interface
    participant VP as VoiceProcessor
    participant SR as Speech Recognition<br/>(Google API)
    participant C as Claude API<br/>(Anthropic)
    participant TTS as Text-to-Speech<br/>(gTTS)
    participant FS as File System

    Note over U,FS: Voice-to-Voice Conversation Flow

    U->>G: 1. Record voice via microphone
    Note over U,G: Audio Format: WAV/WebM<br/>Sample Rate: ~44kHz

    G->>VP: 2. Pass audio file path
    Note over G,VP: File type: filepath<br/>Temporary file location

    VP->>SR: 3. Convert audio to WAV format
    Note over VP,SR: Uses pydub AudioSegment<br/>Ensures compatibility

    VP->>SR: 4. Transcribe audio to text
    SR->>VP: 5. Return transcribed text
    Note over SR,VP: Uses Google's speech-to-text<br/>Language: English (default)

    alt Transcription successful
        VP->>C: 6. Send text to Claude API
        Note over VP,C: Model: claude-sonnet-4-20250514<br/>Max tokens: 120<br/>Temperature: 0.7

        C->>VP: 7. Return AI response text
        Note over C,VP: Optimized for voice:<br/>Conversational & concise

        VP->>TTS: 8. Convert response to speech
        Note over VP,TTS: Language: English<br/>Voice: Default gTTS

        TTS->>FS: 9. Generate audio file
        FS->>VP: 10. Return audio file path
        Note over FS,VP: Format: MP3<br/>Temporary file

        VP->>G: 11. Return audio + text
        G->>U: 12. Play audio response
        Note over G,U: Audio player in browser<br/>Display conversation text

    else Transcription failed
        VP->>TTS: 6b. Generate error message
        TTS->>VP: 7b. Return error audio
        VP->>G: 8b. Return error response
        G->>U: 9b. Play error message
    end

    Note over U,FS: Complete Voice-to-Voice Pipeline<br/>Voice → Text → AI → Text → Voice
'''

def js_btoa(data):
    return base64.b64encode(data)

def pako_deflate(data):
    compress = zlib.compressobj(9, zlib.DEFLATED, 15, 8, zlib.Z_DEFAULT_STRATEGY)
    compressed_data = compress.compress(data)
    compressed_data += compress.flush()
    return compressed_data

def genPakoLink(graphMarkdown: str):
    jGraph = {"code": graphMarkdown, "mermaid": {"theme": "standard"}}
    byteStr = json.dumps(jGraph).encode('utf-8')
    deflated = pako_deflate(byteStr)
    dEncode = js_btoa(deflated)
    link_code = dEncode.decode('ascii')
    return link_code

mermaid_link = genPakoLink(mermaid_code)
print("mermaid.live link:")
print('http://mermaid.live/edit#pako:' + mermaid_link)

mermaid.live link:
http://mermaid.live/edit#pako:eNp1VV2P0kAU/Ss3PPgRgQVk1W3MJsgKbuJqQwFf9mXaXmBiO1Nnprui0d/unZlCWVr7QmnPnXPuuR/93Ulkip0AOvdC448SRYI3nG0Vy+8F0FUwZXjCCyYMrIBpWGlUFx+UfKTfJmRuIXPFUi7hVhhUG5ZgE7YOLW4teYKhkglqLVsOixYWFRWIyQ4WmMit4IZL8T5WF9cv5lJuM4RJePuyGTq1kdOMlalD+IiJMDslC560BCyXkQ1Z4k/TM7LnSX3Ylt61RMxcwIyThmivDZJhHvRFGgT5gApW3VkU+DTtoe4GplLQO81sJjDL5OMhbtW7vp4HMOy7XFUKDw7/wBnkPCHhOymwSUEhk9L6PZMqZyaAb5P1xTeM75z6iOUFKVwwgwH8HY+/f/p14JsT3zoMYNSHkGkNzJ2ysQkVzOzOmeZdC3b5mn1Bp1mkBTqeJeaFVEzt/QGZTFyCB651SGTRIoDX/coAU/EZaQXDxok/51yHXRtEPaeh2Kdl7FONcJujMI74o9CloteJzEkMj3nGzb5BO+7DUjGhE8VjrJkNldsjo0XlxqW135RKgDkGpCfAWly0cI44cb4Zn2vQrnFstW2IU/iZiW3JtuTYR7HNuN7BixQ3rMzMy4NOlpmjvsI1hi4TOxabMvOIYzbTAN70IULhRdks6j6vsU88pJg7GvMsgMRBe1oKgaY37o0Go8vB5XDshN6xn3TcdxSaunA0ONYVFSNDSP+g//ag2F7TyrK3R8smt0C1KCTtkhPLnuqZOte+Upo5/0XWUuV9pweO8HQ8WAbPqLAi4RpPiZ0RNJUBvKvbqSaWVRX+44YLbBTFkbsJDeDGlwfs5J/y0l8itjN91Yc5CmsMnsxNjZxFlTfDwdGc1vl6qm4W+SGrJvkufN0yXA0n7NIYntG8OvPfDvvKlpWGPWP7CnXwrE3LvLs6bJaCIug

## Installation

In [3]:
!pip install gradio speechrecognition pydub gTTS python-speech-features librosa soundfile

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.4 MB/s eta 0:00:00
  Created wheel for python-speech-features: filename=python_speech_features-0.6-py3-none-any.whl size=5868 sha256=95a09a3651ab06e206732f2673388860a9c68eda68687f0e283722f351e374b4
  Stored in directory: /root/.cache/pip/wheels/37/01/19/e6c69a32684ab7b2e3ea4985a571d810cf055c72600e7f9f17
Successfully built python-speech-features
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1


## Echo Bot

In [4]:
# Voice Recording & Playback Gradio App for Google Colab
# Install required packages first:
# !pip install gradio speechrecognition pydub gTTS python-speech-features librosa soundfile

import gradio as gr
import speech_recognition as sr
import tempfile
import os
from gtts import gTTS
import io
import soundfile as sf
import numpy as np
from pydub import AudioSegment
import librosa

class VoiceProcessor:
    def __init__(self):
        self.recognizer = sr.Recognizer()

    def transcribe_audio(self, audio_file):
        """Convert speech to text using SpeechRecognition"""
        try:
            # Convert to wav if needed
            audio = AudioSegment.from_file(audio_file)

            # Export as wav for speech recognition
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_wav:
                audio.export(tmp_wav.name, format="wav")

                # Transcribe using speech recognition
                with sr.AudioFile(tmp_wav.name) as source:
                    audio_data = self.recognizer.record(source)
                    text = self.recognizer.recognize_google(audio_data)

                # Clean up temp file
                os.unlink(tmp_wav.name)
                return text

        except sr.UnknownValueError:
            return "Could not understand audio"
        except sr.RequestError as e:
            return f"Could not request results; {e}"
        except Exception as e:
            return f"Error processing audio: {e}"

    def text_to_speech(self, text, lang='en'):
        """Convert text to speech using gTTS"""
        try:
            # Create TTS object
            tts = gTTS(text=text, lang=lang, slow=False)

            # Save to temporary file
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
                tts.save(tmp_file.name)
                return tmp_file.name

        except Exception as e:
            print(f"Error in TTS: {e}")
            return None

def process_voice_input(audio_file):
    """Main function to process voice input and return voice output"""
    if audio_file is None:
        return None, "No audio provided"

    processor = VoiceProcessor()

    # Transcribe the input audio
    transcribed_text = processor.transcribe_audio(audio_file)

    # Create a response (you can modify this logic)
    if "hello" in transcribed_text.lower():
        response_text = f"Hello! You said: '{transcribed_text}'. Nice to meet you!"
    elif "how are you" in transcribed_text.lower():
        response_text = f"I'm doing well, thank you! You asked: '{transcribed_text}'"
    else:
        response_text = f"I heard you say: '{transcribed_text}'. Thank you for your message!"

    # Convert response to speech
    output_audio = processor.text_to_speech(response_text)

    return output_audio, f"Transcribed: {transcribed_text}\nResponse: {response_text}"

def echo_audio(audio_file):
    """Simple echo function - returns the same audio"""
    if audio_file is None:
        return None, "No audio provided"
    return audio_file, "Audio echoed back"

def analyze_audio_features(audio_file):
    """Analyze audio features and provide feedback"""
    if audio_file is None:
        return None, "No audio provided"

    try:
        # Load audio file
        y, sr_rate = librosa.load(audio_file)

        # Extract features
        duration = len(y) / sr_rate
        rms_energy = librosa.feature.rms(y=y)[0].mean()
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr_rate)[0].mean()

        # Create analysis text
        analysis = f"""
        Audio Analysis:
        - Duration: {duration:.2f} seconds
        - Average Energy: {rms_energy:.4f}
        - Spectral Centroid: {spectral_centroid:.2f} Hz
        - Sample Rate: {sr_rate} Hz
        """

        # Convert analysis to speech
        processor = VoiceProcessor()
        analysis_speech = f"Audio analysis complete. Duration is {duration:.1f} seconds with moderate energy levels."
        output_audio = processor.text_to_speech(analysis_speech)

        return output_audio, analysis

    except Exception as e:
        return None, f"Error analyzing audio: {e}"

# Create Gradio interface
def create_voice_app():
    with gr.Blocks(title="Voice Recording & Playback App") as app:
        gr.Markdown("# 🎤 Voice Recording & Playback Application")
        gr.Markdown("Record your voice and get intelligent responses back!")

        with gr.Tab("Speech-to-Speech Chat"):
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(
                        label="Record Your Voice",
                        type="filepath"
                    )
                    process_btn = gr.Button("Process Voice", variant="primary")

                with gr.Column():
                    audio_output = gr.Audio(label="AI Response")
                    text_output = gr.Textbox(
                        label="Transcription & Response",
                        lines=4,
                        interactive=False
                    )

            process_btn.click(
                fn=process_voice_input,
                inputs=[audio_input],
                outputs=[audio_output, text_output]
            )

        with gr.Tab("Audio Echo"):
            with gr.Row():
                with gr.Column():
                    echo_input = gr.Audio(
                        label="Record Audio to Echo",
                        type="filepath"
                    )
                    echo_btn = gr.Button("Echo Audio", variant="secondary")

                with gr.Column():
                    echo_output = gr.Audio(label="Echoed Audio")
                    echo_text = gr.Textbox(
                        label="Status",
                        interactive=False
                    )

            echo_btn.click(
                fn=echo_audio,
                inputs=[echo_input],
                outputs=[echo_output, echo_text]
            )

        with gr.Tab("Audio Analysis"):
            with gr.Row():
                with gr.Column():
                    analyze_input = gr.Audio(
                        label="Record Audio for Analysis",
                        type="filepath"
                    )
                    analyze_btn = gr.Button("Analyze Audio", variant="secondary")

                with gr.Column():
                    analyze_audio_out = gr.Audio(label="Analysis Speech")
                    analyze_text_out = gr.Textbox(
                        label="Audio Features",
                        lines=8,
                        interactive=False
                    )

            analyze_btn.click(
                fn=analyze_audio_features,
                inputs=[analyze_input],
                outputs=[analyze_audio_out, analyze_text_out]
            )

        with gr.Tab("Instructions"):
            gr.Markdown("""
            ## How to Use:

            ### Speech-to-Speech Chat:
            1. Click the microphone icon to start recording
            2. Speak clearly into your microphone
            3. Click "Process Voice" to get a spoken response

            ### Audio Echo:
            1. Record any audio (voice, music, etc.)
            2. Click "Echo Audio" to hear it played back

            ### Audio Analysis:
            1. Record audio for technical analysis
            2. Get both spoken and written feedback about audio characteristics

            ### Tips:
            - Ensure your microphone is enabled in your browser
            - Speak clearly and avoid background noise
            - Wait for processing to complete before recording again
            """)

    return app

# Installation code for Colab
colab_setup = '''
# Run this cell first in Google Colab:
!pip install gradio speechrecognition pydub gTTS librosa soundfile pyaudio

# If you get pyaudio errors, try:
# !apt-get install portaudio19-dev python3-pyaudio
# !pip install pyaudio
'''

print("🎤 Voice Recording & Playback Gradio App")
print("=" * 50)
print("Setup Instructions for Google Colab:")
print(colab_setup)
print("=" * 50)

# Create and launch the app
if __name__ == "__main__":
    app = create_voice_app()
    app.launch(
        share=True,  # Creates public link for Colab
        debug=True,
        server_name="0.0.0.0",  # Important for Colab
        server_port=7860
    )

🎤 Voice Recording & Playback Gradio App
Setup Instructions for Google Colab:

# Run this cell first in Google Colab:
!pip install gradio speechrecognition pydub gTTS librosa soundfile pyaudio

# If you get pyaudio errors, try:
# !apt-get install portaudio19-dev python3-pyaudio
# !pip install pyaudio

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f42d5b6a03021cf18c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://f42d5b6a03021cf18c.gradio.live


## GPT-2 (from HF)

In [5]:
# Voice Recording & Playback Gradio App for Google Colab
# Install required packages first:
# !pip install gradio speechrecognition pydub gTTS python-speech-features librosa soundfile transformers torch

import gradio as gr
import speech_recognition as sr
import tempfile
import os
from gtts import gTTS
import io
import soundfile as sf
import numpy as np
from pydub import AudioSegment
import librosa
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

class VoiceProcessor:
    def __init__(self):
        self.recognizer = sr.Recognizer()
        # Load GPT-2 Large model for text generation
        print("Loading GPT-2 Large model...")
        self.tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2-large')
        self.model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2-large')

        # Add padding token if it doesn't exist
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Set device
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        print(f"Model loaded on {self.device}")

    def generate_response(self, input_text, max_length=100, temperature=0.8, top_p=0.9):
        """Generate intelligent response using GPT-2 Large"""
        try:
            # Create a conversational prompt
            prompt = f"Human: {input_text}\nAI:"

            # Tokenize input
            inputs = self.tokenizer.encode(prompt, return_tensors='pt').to(self.device)

            # Generate response
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs,
                    max_length=inputs.shape[1] + max_length,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id,
                    num_return_sequences=1,
                    repetition_penalty=1.1
                )

            # Decode and extract response
            full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Extract just the AI response part
            if "AI:" in full_response:
                ai_response = full_response.split("AI:")[-1].strip()
                # Clean up the response (remove potential "Human:" if it appears)
                if "Human:" in ai_response:
                    ai_response = ai_response.split("Human:")[0].strip()
            else:
                ai_response = full_response[len(prompt):].strip()

            # Ensure response isn't too long or empty
            if not ai_response or len(ai_response) < 5:
                ai_response = f"That's interesting! You mentioned: {input_text}. Could you tell me more about that?"

            # Truncate if too long
            if len(ai_response) > 300:
                ai_response = ai_response[:300] + "..."

            return ai_response

        except Exception as e:
            print(f"Error in text generation: {e}")
            return f"I heard you say '{input_text}'. That's quite interesting! Could you elaborate on that?"

    def transcribe_audio(self, audio_file):
        """Convert speech to text using SpeechRecognition"""
        try:
            # Convert to wav if needed
            audio = AudioSegment.from_file(audio_file)

            # Export as wav for speech recognition
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_wav:
                audio.export(tmp_wav.name, format="wav")

                # Transcribe using speech recognition
                with sr.AudioFile(tmp_wav.name) as source:
                    audio_data = self.recognizer.record(source)
                    text = self.recognizer.recognize_google(audio_data)

                # Clean up temp file
                os.unlink(tmp_wav.name)
                return text

        except sr.UnknownValueError:
            return "Could not understand audio"
        except sr.RequestError as e:
            return f"Could not request results; {e}"
        except Exception as e:
            return f"Error processing audio: {e}"

    def text_to_speech(self, text, lang='en'):
        """Convert text to speech using gTTS"""
        try:
            # Create TTS object
            tts = gTTS(text=text, lang=lang, slow=False)

            # Save to temporary file
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
                tts.save(tmp_file.name)
                return tmp_file.name

        except Exception as e:
            print(f"Error in TTS: {e}")
            return None

# Global processor instance to avoid reloading model
global_processor = None

def get_processor():
    """Get or create the global processor instance"""
    global global_processor
    if global_processor is None:
        global_processor = VoiceProcessor()
    return global_processor

def process_voice_input(audio_file):
    """Main function to process voice input and return voice output"""
    if audio_file is None:
        return None, "No audio provided"

    processor = get_processor()

    # Transcribe the input audio
    transcribed_text = processor.transcribe_audio(audio_file)

    # Generate intelligent response using GPT-2
    response_text = processor.generate_response(transcribed_text)

    # Convert response to speech
    output_audio = processor.text_to_speech(response_text)

    return output_audio, f"You said: '{transcribed_text}'\n\nAI Response: {response_text}"

def llm_conversation(audio_file):
    """LLM-powered conversation using GPT-2 Large"""
    if audio_file is None:
        return None, "No audio provided"

    processor = get_processor()

    # Transcribe the input audio
    transcribed_text = processor.transcribe_audio(audio_file)

    if transcribed_text in ["Could not understand audio", "Could not request results"]:
        response_text = "I'm sorry, I couldn't understand your audio clearly. Could you please try again?"
    else:
        # Generate intelligent response using GPT-2
        response_text = processor.generate_response(
            transcribed_text,
            max_length=80,  # Shorter for voice responses
            temperature=0.7,  # Slightly more focused
            top_p=0.85
        )

    # Convert response to speech
    output_audio = processor.text_to_speech(response_text)

    return output_audio, f"Transcription: {transcribed_text}\n\nGPT-2 Response: {response_text}"

def analyze_audio_features(audio_file):
    """Analyze audio features and provide feedback"""
    if audio_file is None:
        return None, "No audio provided"

    try:
        # Load audio file
        y, sr_rate = librosa.load(audio_file)

        # Extract features
        duration = len(y) / sr_rate
        rms_energy = librosa.feature.rms(y=y)[0].mean()
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr_rate)[0].mean()

        # Create analysis text
        analysis = f"""
        Audio Analysis:
        - Duration: {duration:.2f} seconds
        - Average Energy: {rms_energy:.4f}
        - Spectral Centroid: {spectral_centroid:.2f} Hz
        - Sample Rate: {sr_rate} Hz
        """

        # Convert analysis to speech
        processor = get_processor()
        analysis_speech = f"Audio analysis complete. Duration is {duration:.1f} seconds with moderate energy levels."
        output_audio = processor.text_to_speech(analysis_speech)

        return output_audio, analysis

    except Exception as e:
        return None, f"Error analyzing audio: {e}"

# Create Gradio interface
def create_voice_app():
    with gr.Blocks(title="Voice Recording & Playback App") as app:
        gr.Markdown("# 🎤 Voice Recording & Playback Application")
        gr.Markdown("Record your voice and get intelligent responses back!")

        with gr.Tab("Speech-to-Speech Chat"):
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(
                        label="Record Your Voice",
                        type="filepath"
                    )
                    process_btn = gr.Button("Process Voice", variant="primary")

                with gr.Column():
                    audio_output = gr.Audio(label="AI Response")
                    text_output = gr.Textbox(
                        label="Transcription & Response",
                        lines=4,
                        interactive=False
                    )

            process_btn.click(
                fn=process_voice_input,
                inputs=[audio_input],
                outputs=[audio_output, text_output]
            )

        with gr.Tab("LLM Conversation"):
            with gr.Row():
                with gr.Column():
                    llm_input = gr.Audio(
                        label="Talk to GPT-2 Large",
                        type="filepath"
                    )
                    llm_btn = gr.Button("Chat with AI", variant="secondary")

                with gr.Column():
                    llm_output = gr.Audio(label="AI Voice Response")
                    llm_text = gr.Textbox(
                        label="Conversation",
                        lines=6,
                        interactive=False
                    )

            llm_btn.click(
                fn=llm_conversation,
                inputs=[llm_input],
                outputs=[llm_output, llm_text]
            )

        with gr.Tab("Audio Analysis"):
            with gr.Row():
                with gr.Column():
                    analyze_input = gr.Audio(
                        label="Record Audio for Analysis",
                        type="filepath"
                    )
                    analyze_btn = gr.Button("Analyze Audio", variant="secondary")

                with gr.Column():
                    analyze_audio_out = gr.Audio(label="Analysis Speech")
                    analyze_text_out = gr.Textbox(
                        label="Audio Features",
                        lines=8,
                        interactive=False
                    )

            analyze_btn.click(
                fn=analyze_audio_features,
                inputs=[analyze_input],
                outputs=[analyze_audio_out, analyze_text_out]
            )

        with gr.Tab("Instructions"):
            gr.Markdown("""
            ## How to Use:

            ### Speech-to-Speech Chat:
            1. Click the microphone icon to start recording
            2. Speak clearly into your microphone
            3. Click "Process Voice" to get a spoken response

            ### LLM Conversation:
            1. Record your voice with any question or statement
            2. Click "Chat with AI" to get an intelligent GPT-2 Large response
            3. The AI will provide contextual, conversational responses

            ### Audio Analysis:
            1. Record audio for technical analysis
            2. Get both spoken and written feedback about audio characteristics

            ### Tips:
            - Ensure your microphone is enabled in your browser
            - Speak clearly and avoid background noise
            - Wait for processing to complete before recording again
            - The LLM may take a few seconds to generate intelligent responses
            - Try asking questions, making statements, or having casual conversations

            ### Model Information:
            - Using GPT-2 Large (774M parameters) from Hugging Face
            - Supports natural conversation and contextual responses
            - Optimized for voice interaction with appropriate response lengths
            """)

    return app

# Installation code for Colab
colab_setup = '''
# Run this cell first in Google Colab:
!pip install gradio speechrecognition pydub gTTS librosa soundfile transformers torch

# If you get pyaudio errors, try:
# !apt-get install portaudio19-dev python3-pyaudio
# !pip install pyaudio
'''

print("🎤 Voice Recording & Playback Gradio App")
print("=" * 50)
print("Setup Instructions for Google Colab:")
print(colab_setup)
print("=" * 50)

# Create and launch the app
if __name__ == "__main__":
    app = create_voice_app()
    app.launch(
        share=True,  # Creates public link for Colab
        debug=True,
        server_name="0.0.0.0",  # Important for Colab
        server_port=7860
    )

🎤 Voice Recording & Playback Gradio App
Setup Instructions for Google Colab:

# Run this cell first in Google Colab:
!pip install gradio speechrecognition pydub gTTS librosa soundfile transformers torch

# If you get pyaudio errors, try:
# !apt-get install portaudio19-dev python3-pyaudio
# !pip install pyaudio

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6e98d335fb36a43543.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading GPT-2 Large model...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded on cuda


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://6e98d335fb36a43543.gradio.live


## Anthropic (API Key required)

In [6]:
pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 7.9 MB/s eta 0:00:00


In [7]:
from google.colab import userdata
CLAUDE_API_KEY = userdata.get('CLAUDE_API_KEY')

In [ ]:
# Voice Recording & Playback Gradio App for Google Colab with Claude Integration
# Install required packages first:
# !pip install gradio speechrecognition pydub gTTS python-speech-features librosa soundfile anthropic

import gradio as gr
import speech_recognition as sr
import tempfile
import os
from gtts import gTTS
import io
import soundfile as sf
import numpy as np
from pydub import AudioSegment
import librosa
import anthropic

class VoiceProcessor:
    def __init__(self):
        self.recognizer = sr.Recognizer()

        # Initialize Claude client
        api_key = os.getenv('CLAUDE_API_KEY', CLAUDE_API_KEY)
        if not api_key:
            raise ValueError("CLAUDE_API_KEY environment variable not found. Please set it in Colab.")

        self.claude_client = anthropic.Anthropic(api_key=api_key)
        print("Claude API client initialized successfully!")

    def generate_response(self, input_text, max_tokens=150, temperature=0.7):
        """Generate intelligent response using Claude API"""
        try:
            # Create a conversational prompt
            system_prompt = """You are a helpful, friendly AI assistant engaged in voice conversation.
            Keep your responses conversational, concise (under 2-3 sentences), and engaging since they will be converted to speech.
            Be natural and personable while being informative."""

            message = self.claude_client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=max_tokens,
                temperature=temperature,
                system=system_prompt,
                messages=[
                    {
                        "role": "user",
                        "content": input_text
                    }
                ]
            )

            # Extract the response text
            response_text = message.content[0].text.strip()

            # Ensure response isn't too long for voice synthesis
            if len(response_text) > 400:
                response_text = response_text[:400] + "..."

            return response_text

        except anthropic.APIError as e:
            print(f"Claude API Error: {e}")
            return f"I'm having trouble connecting to my AI service right now. You mentioned: '{input_text}'. Could you try again?"
        except Exception as e:
            print(f"Error in Claude generation: {e}")
            return f"I heard you say '{input_text}'. That's interesting! Could you tell me more about that?"

    def transcribe_audio(self, audio_file):
        """Convert speech to text using SpeechRecognition"""
        try:
            # Convert to wav if needed
            audio = AudioSegment.from_file(audio_file)

            # Export as wav for speech recognition
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_wav:
                audio.export(tmp_wav.name, format="wav")

                # Transcribe using speech recognition
                with sr.AudioFile(tmp_wav.name) as source:
                    audio_data = self.recognizer.record(source)
                    text = self.recognizer.recognize_google(audio_data)

                # Clean up temp file
                os.unlink(tmp_wav.name)
                return text

        except sr.UnknownValueError:
            return "Could not understand audio"
        except sr.RequestError as e:
            return f"Could not request results; {e}"
        except Exception as e:
            return f"Error processing audio: {e}"

    def text_to_speech(self, text, lang='en'):
        """Convert text to speech using gTTS"""
        try:
            # Create TTS object
            tts = gTTS(text=text, lang=lang, slow=False)

            # Save to temporary file
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
                tts.save(tmp_file.name)
                return tmp_file.name

        except Exception as e:
            print(f"Error in TTS: {e}")
            return None

# Global processor instance to avoid reloading model
global_processor = None

def get_processor():
    """Get or create the global processor instance"""
    global global_processor
    if global_processor is None:
        global_processor = VoiceProcessor()
    return global_processor

def process_voice_input(audio_file):
    """Main function to process voice input and return voice output"""
    if audio_file is None:
        return None, "No audio provided"

    processor = get_processor()

    # Transcribe the input audio
    transcribed_text = processor.transcribe_audio(audio_file)

    # Generate intelligent response using Claude
    response_text = processor.generate_response(transcribed_text)

    # Convert response to speech
    output_audio = processor.text_to_speech(response_text)

    return output_audio, f"You said: '{transcribed_text}'\n\nClaude: {response_text}"

def llm_conversation(audio_file):
    """LLM-powered conversation using Claude API"""
    if audio_file is None:
        return None, "No audio provided"

    processor = get_processor()

    # Transcribe the input audio
    transcribed_text = processor.transcribe_audio(audio_file)

    if transcribed_text in ["Could not understand audio", "Could not request results"]:
        response_text = "I'm sorry, I couldn't understand your audio clearly. Could you please try again?"
    else:
        # Generate intelligent response using Claude
        response_text = processor.generate_response(
            transcribed_text,
            max_tokens=120,  # Appropriate for voice responses
            temperature=0.7  # Good balance for conversation
        )

    # Convert response to speech
    output_audio = processor.text_to_speech(response_text)

    return output_audio, f"You: {transcribed_text}\n\nClaude: {response_text}"

def analyze_audio_features(audio_file):
    """Analyze audio features and provide feedback"""
    if audio_file is None:
        return None, "No audio provided"

    try:
        # Load audio file
        y, sr_rate = librosa.load(audio_file)

        # Extract features
        duration = len(y) / sr_rate
        rms_energy = librosa.feature.rms(y=y)[0].mean()
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr_rate)[0].mean()

        # Create analysis text
        analysis = f"""
        Audio Analysis:
        - Duration: {duration:.2f} seconds
        - Average Energy: {rms_energy:.4f}
        - Spectral Centroid: {spectral_centroid:.2f} Hz
        - Sample Rate: {sr_rate} Hz
        """

        # Convert analysis to speech
        processor = get_processor()
        analysis_speech = f"Audio analysis complete. Duration is {duration:.1f} seconds with moderate energy levels."
        output_audio = processor.text_to_speech(analysis_speech)

        return output_audio, analysis

    except Exception as e:
        return None, f"Error analyzing audio: {e}"

# Create Gradio interface
def create_voice_app():
    with gr.Blocks(title="Voice Recording & Playback App") as app:
        gr.Markdown("# 🎤 Voice Recording & Playback Application")
        gr.Markdown("Record your voice and get intelligent responses back!")

        with gr.Tab("Speech-to-Speech Chat"):
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(
                        label="Record Your Voice",
                        type="filepath"
                    )
                    process_btn = gr.Button("Process Voice", variant="primary")

                with gr.Column():
                    audio_output = gr.Audio(label="AI Response")
                    text_output = gr.Textbox(
                        label="Transcription & Response",
                        lines=4,
                        interactive=False
                    )

            process_btn.click(
                fn=process_voice_input,
                inputs=[audio_input],
                outputs=[audio_output, text_output]
            )

        with gr.Tab("Claude Conversation"):
            with gr.Row():
                with gr.Column():
                    llm_input = gr.Audio(
                        label="Talk to Claude",
                        type="filepath"
                    )
                    llm_btn = gr.Button("Chat with Claude", variant="secondary")

                with gr.Column():
                    llm_output = gr.Audio(label="Claude Voice Response")
                    llm_text = gr.Textbox(
                        label="Conversation",
                        lines=6,
                        interactive=False
                    )

            llm_btn.click(
                fn=llm_conversation,
                inputs=[llm_input],
                outputs=[llm_output, llm_text]
            )

        with gr.Tab("Audio Analysis"):
            with gr.Row():
                with gr.Column():
                    analyze_input = gr.Audio(
                        label="Record Audio for Analysis",
                        type="filepath"
                    )
                    analyze_btn = gr.Button("Analyze Audio", variant="secondary")

                with gr.Column():
                    analyze_audio_out = gr.Audio(label="Analysis Speech")
                    analyze_text_out = gr.Textbox(
                        label="Audio Features",
                        lines=8,
                        interactive=False
                    )

            analyze_btn.click(
                fn=analyze_audio_features,
                inputs=[analyze_input],
                outputs=[analyze_audio_out, analyze_text_out]
            )

        with gr.Tab("Instructions"):
            gr.Markdown("""
            ## How to Use:

            ### Speech-to-Speech Chat:
            1. Click the microphone icon to start recording
            2. Speak clearly into your microphone
            3. Click "Process Voice" to get a spoken response from Claude

            ### Claude Conversation:
            1. Record your voice with any question or statement
            2. Click "Chat with Claude" to get an intelligent response
            3. Claude will provide contextual, conversational responses via API

            ### Audio Analysis:
            1. Record audio for technical analysis
            2. Get both spoken and written feedback about audio characteristics

            ### Tips:
            - Ensure your microphone is enabled in your browser
            - Speak clearly and avoid background noise
            - Wait for processing to complete before recording again
            - Claude API responses may take a few seconds
            - Try asking questions, making statements, or having casual conversations

            ### Setup Requirements (Colab):
            ```python
            import os
            os.environ['CLAUDE_API_KEY'] = 'your-claude-api-key-here'
            ```

            ### Model Information:
            - Using Claude Sonnet 4 via Anthropic API
            - Supports natural conversation and contextual responses
            - Optimized for voice interaction with appropriate response lengths
            - Requires valid CLAUDE_API_KEY environment variable
            """)

    return app

# Installation code for Colab
colab_setup = '''
# Run this cell first in Google Colab:
!pip install gradio speechrecognition pydub gTTS librosa soundfile anthropic

# Set your Claude API key:
import os
os.environ['CLAUDE_API_KEY'] = 'your-claude-api-key-here'  # Replace with your actual key

# If you get pyaudio errors, try:
# !apt-get install portaudio19-dev python3-pyaudio
# !pip install pyaudio
'''

print("🎤 Voice Recording & Playback Gradio App")
print("=" * 50)
print("Setup Instructions for Google Colab:")
print(colab_setup)
print("=" * 50)

# Create and launch the app
if __name__ == "__main__":
    app = create_voice_app()
    app.launch(
        share=True,  # Creates public link for Colab
        debug=True,
        server_name="0.0.0.0",  # Important for Colab
        server_port=7860
    )

🎤 Voice Recording & Playback Gradio App
Setup Instructions for Google Colab:

# Run this cell first in Google Colab:
!pip install gradio speechrecognition pydub gTTS librosa soundfile anthropic

# Set your Claude API key:
import os
os.environ['CLAUDE_API_KEY'] = 'your-claude-api-key-here'  # Replace with your actual key

# If you get pyaudio errors, try:
# !apt-get install portaudio19-dev python3-pyaudio
# !pip install pyaudio

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cd6fa8e24621953e8d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Claude API client initialized successfully!
